# 05 — PPI controls (Table 5, S4), per-gene tables, slide / cancer-type stratification, final comparison table
Reproduces the network audit, the degree-preserving rewired graphs (from the seeds) and the edge-free control, then re-evaluates the six control checkpoints on the locked test and external-HD cohorts (25% random and neighbourhood masking), and rebuilds the per-gene MSE tables, the slide-/cancer-level Full-vs-Ring2Mean/expression+spatial table and the final comparison table. Every regenerated CSV is compared with the stored one. No training.
The physical-subnetwork count (126/680) and the functional-network cross-check come from a live STRING query (current server state; may drift) and are reported, not asserted.

In [ ]:
# --- Setup: mount Drive, unpack the repository, install dependencies ---
from google.colab import drive
drive.mount('/content/drive')
import subprocess, sys, os
BASE = '/content/drive/MyDrive/New-SASP-idea'                 # data/checkpoints live here (not in the repo)
REPO_ZIP = BASE + '/code_release_export_2026-10-23/spatial-sasp-gat.zip'   # or: git clone <repo URL> /content/spatial-sasp-gat
import shutil, hashlib
if os.path.isdir('/content/spatial-sasp-gat'):
    shutil.rmtree('/content/spatial-sasp-gat')          # never reuse an older unpacked copy: always run the zip that is on Drive
subprocess.run(['unzip', '-q', '-o', REPO_ZIP, '-d', '/content'], check=True)
print('repository zip sha256[:16]:', hashlib.sha256(open(REPO_ZIP, 'rb').read()).hexdigest()[:16], '| size', os.path.getsize(REPO_ZIP))
if 'sasp_gat' in sys.modules: print('WARNING: sasp_gat was already imported in this runtime - reset the runtime so the new code is used')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'anndata'], check=True)   # reads HEST .h5ad; scikit-learn / networkx are preinstalled on Colab (do NOT reinstall: it can swap pandas under a running kernel)
# data_tables exported in ENV-2/ENV-3 (copied only if missing in the repo)
import shutil
for f in ['sasp54_genes_canonical_order.csv', 'ppi_string_v12_edges.csv', 'slide_partition.csv']:
    dst = '/content/spatial-sasp-gat/data_tables/' + f
    if not os.path.exists(dst):
        shutil.copy(BASE + '/code_release_export_2026-10-23/' + f, dst)
sys.path.insert(0, '/content/spatial-sasp-gat/src')
import torch, torch_geometric
print('torch', torch.__version__, '| torch_geometric', torch_geometric.__version__,
      '| cuda', torch.cuda.is_available())

In [ ]:
import hashlib, json, pandas as pd, numpy as np, torch
from pathlib import Path
from sasp_gat.config import *
from sasp_gat.data import load_package, split_samples, sha256_file
from sasp_gat.ppi import ppi_edge_index_from_networkx
from sasp_gat.availability import build_availability, effective_mask, score_mask
from sasp_gat.evaluation import load_model, eval_both
from sasp_gat.training import evaluate_multimask_av

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
REPO = Path('/content/spatial-sasp-gat'); T = REPO / 'data_tables'
PKL = Path(BASE) / 'MODEL_READY_DATASET_v1_2026-09-26.pkl'
pkg, pyg_dataset, gene_order = load_package(PKL)
train_s, val_s, test_s, hd_s = split_samples(pyg_dataset)
AVAIL, LOGGED = build_availability(pyg_dataset.keys(), gene_order, T / 'availability_unmeasured.csv')
ppi_ei = ppi_edge_index_from_networkx(pkg['ppi_graph_networkx'], gene_order, device)
CKDIR = REPO / 'checkpoints' if (REPO / 'checkpoints' / 'full_avail_seed42.pt').exists() else Path(BASE) / 'revision_checkpoints_2026-10-18'
print('checkpoint dir:', CKDIR)
import torch_geometric, scipy; print('versions: torch', torch.__version__, '| pyg', torch_geometric.__version__, '| numpy', np.__version__, '| pandas', pd.__version__, '| scipy', scipy.__version__)
import warnings; warnings.filterwarnings('ignore')
ABL_DIR = Path(BASE) / 'revision_checkpoints_2026-10-18'
TAB_DIR = Path(BASE) / 'revision_tables_2026-10-18'
MR_DIR = Path(BASE) / 'revision_masking_robustness_2026-10-19'
CTRL_DIR = Path(BASE) / 'revision_ppi_controls_2026-10-19'
PG_DIR = Path(BASE) / 'revision_per_gene_2026-10-19'
AUD_DIR = Path(BASE) / 'revision_audit_2026-10-19'
FIN_DIR = Path(BASE) / 'revision_final_tables_2026-10-19'
OUT_DIR = Path(BASE) / 'repro_outputs_2026-10-24'   # regenerated tables (never overwrites the revision folders)
OUT_DIR.mkdir(exist_ok=True)


In [ ]:
from sasp_gat.evaluation import load_all_models, VARIANT_KEYS
from sasp_gat.twohop import build_graph_cache
from sasp_gat.baselines import train_gene_means
from sasp_gat.scenarios import Predictors
from sasp_gat.verify import compare_csv, report_compare
from sasp_gat import ppi_controls as pc, tables as tb
MODELS = load_all_models(CKDIR, device, variants=['full'])
MODELS.update(load_all_models(ABL_DIR, device, variants=[v for v in VARIANT_KEYS if v != 'full']))
mu32 = train_gene_means(train_s, AVAIL)
GR = build_graph_cache({'val': val_s, 'test': test_s, 'hd': hd_s}, device)
pred = Predictors(MODELS, ppi_ei, mu32, device)
# scenario table: regenerated by notebook 04 if present, else the stored one
f_res = OUT_DIR / 'res17_per_slide_mask_seed.csv'
RES17 = pd.read_csv(f_res if f_res.exists() else MR_DIR / 'res17_per_slide_mask_seed.csv')
print('RES17 from', 'regenerated (notebook 04)' if f_res.exists() else 'STORED revision file', '| rows', len(RES17))

In [ ]:
# 1) Network audit + live STRING cross-check
real = pc.real_undirected_edges(pkg['ppi_graph_networkx'], gene_order)
AUD, deg = pc.network_audit(real); print(AUD)
ei_real = pc.to_edge_index(real, device)
assert set(map(tuple, ei_real.t().tolist())) == set(map(tuple, ppi_ei.t().tolist())); print('edge index == the one used in training')
STR = {}
for nt in ['functional', 'physical']:
    try:
        e, nrow, unm = pc.string_network(gene_order, nt); STR[nt] = e
        print(f'{nt:10s}: API {len(e)} edges | in stored network {len(e & set(real))} | only API {len(e - set(real))} | only stored {len(set(real) - e)} | unmapped names {len(unm)}')
    except Exception as ex:
        print(nt, 'API unavailable:', type(ex).__name__, str(ex)[:150])
if 'functional' in STR: print('stored network == current STRING functional @400:', STR['functional'] == set(real))
if len(STR) == 2: print('physical subset of functional:', STR['physical'] <= STR['functional'], '| physical edges', len(STR['physical']))

In [ ]:
# 2) Control graphs regenerated from the seeds must equal the graphs stored in the control checkpoints (hash), then evaluate
REW = pc.rewired_graphs(real, deg, device)
GRAPHS = {}
for s in SEEDS:
    GRAPHS[('edge_free', s)] = torch.empty((2, 0), dtype=torch.long, device=device)
    GRAPHS[('rewired', s)] = REW[s]['ei']
print('overlap with real edges (rewired, per seed):', {s: REW[s]['overlap'] for s in SEEDS}, '/ 680')
CM = pc.load_control_models(CTRL_DIR, device, GRAPHS)       # asserts graph hash == regenerated graph, params, flags
print('6 control checkpoints loaded; regenerated rewired graphs are identical to the trained ones')
rows = []
pc.evaluate_controls('val25', ('rand', 0.25), 'val', val_s, 'val', MODELS, CM, ppi_ei, AVAIL, GR, device, rows)
g0 = pd.DataFrame(rows)
for (c, s), (_, _, bv) in CM.items():
    z = g0[(g0.method == pc.NAME[c]) & (g0.seed == s)]
    assert abs(z.ss.sum() / z.n.sum() - bv) < 1e-5, (c, s)
print('gate 1: validation MSE of all 6 controls reproduces best_val_loss (<1e-5)')
R2 = []
for scn_name, scn in [('rand25', ('rand', 0.25)), ('nbhd05', ('nbhd', 0.05))]:
    pc.evaluate_controls(scn_name, scn, 'test', test_s, 'test', MODELS, CM, ppi_ei, AVAIL, GR, device, R2)
    pc.evaluate_controls(scn_name, scn, 'hd', hd_s, 'hd', MODELS, CM, ppi_ei, AVAIL, GR, device, R2)
R2 = pd.DataFrame(R2)
for scn_name in ['rand25', 'nbhd05']:
    for coh in ['test', 'hd']:
        for s in SEEDS:
            a = R2[(R2.scn == scn_name) & (R2.cohort == coh) & (R2.method == pc.REAL_NAME) & (R2.seed == s)]
            b = RES17[(RES17.scn == scn_name) & (RES17.cohort == coh) & (RES17.method == 'full') & (RES17.seed == s)]
            assert abs(a.ss.sum() / a.n.sum() - b.ss.sum() / b.n.sum()) < 1e-5
print('gate 2: real-STRING full reproduces the scenario results')
ref_rows = RES17[RES17.method.isin(['expression-only', 'expression+spatial']) & RES17.scn.isin(['rand25', 'nbhd05']) & RES17.cohort.isin(['test', 'hd'])]
S_ctrl, P_ctrl = pc.summarize_controls(R2, ref_rows)
print(S_ctrl.round(4).to_string(index=False)); print(P_ctrl.round(4).to_string(index=False))
res = {}
for nm, df in {'eval_ppi_controls_per_slide_mask_seed.csv': R2, 'eval_ppi_controls_summary.csv': S_ctrl, 'eval_ppi_controls_paired_vs_real.csv': P_ctrl}.items():
    res[nm] = compare_csv(df, CTRL_DIR / nm) if (CTRL_DIR / nm).exists() else (False, float('nan'), 'stored file not found')
report_compare(res)
for nm, df in {'eval_ppi_controls_per_slide_mask_seed.csv': R2, 'eval_ppi_controls_summary.csv': S_ctrl, 'eval_ppi_controls_paired_vs_real.csv': P_ctrl}.items():
    df.to_csv(OUT_DIR / nm, index=False)

In [ ]:
# 3) Per-gene MSE tables (25% random masks) -> compare with revision_per_gene_2026-10-19
LONG = tb.per_gene_long(pred, test_s, hd_s, AVAIL, GR, gene_order, device)
tb.per_gene_gate(LONG, RES17); print('gate: gene sums reproduce pooled MSE and target counts of the scenario run')
LONG2, SUMG = tb.per_gene_summary(LONG)
res = {}
for nm, df in {'per_gene_long_rand25.csv': LONG2, 'per_gene_summary_rand25.csv': SUMG}.items():
    res[nm] = compare_csv(df, PG_DIR / nm) if (PG_DIR / nm).exists() else (False, float('nan'), 'stored file not found')
report_compare(res)
for coh in ['test', 'hd']:
    W = tb.per_gene_wide(SUMG, coh); print(f'\n{coh.upper()}: full<A {int((W.MSE_full_mean < W.MSE_baselineA).sum())}/54 | full<expr+spatial {int(W.full_lt_expr_spatial.sum())}/54 | full<Ring2Mean {int(W.full_lt_Ring2Mean.sum())}/54 | median rel. improvement {W.rel_improve_vs_A_pct.median():.1f}%')
    print(W.head(5).round(4).to_string())
LONG2.to_csv(OUT_DIR / 'per_gene_long_rand25.csv', index=False); SUMG.to_csv(OUT_DIR / 'per_gene_summary_rand25.csv', index=False)

In [ ]:
# 4) Slide- and cancer-type-level Full vs Ring2Mean / expression+spatial; cancer label from the dataset package (Data.cancer)
cancer_map = {sid: str(d.cancer) for sid, d in {**test_s, **hd_s}.items()}
SL, CANC = tb.slide_level(RES17, cancer_map)
res = {'slide_level_full_vs_comparators.csv': compare_csv(SL, AUD_DIR / 'slide_level_full_vs_comparators.csv') if (AUD_DIR / 'slide_level_full_vs_comparators.csv').exists() else (False, float('nan'), 'stored file not found'),
       'cancer_type_summary.csv': compare_csv(CANC, AUD_DIR / 'cancer_type_summary.csv') if (AUD_DIR / 'cancer_type_summary.csv').exists() else (False, float('nan'), 'stored file not found')}
report_compare(res)
print(CANC.round(4).to_string(index=False))
SL.to_csv(OUT_DIR / 'slide_level_full_vs_comparators.csv', index=False); CANC.to_csv(OUT_DIR / 'cancer_type_summary.csv', index=False)

In [ ]:
# 5) Final comparison table (no new evaluation: merges the regenerated per-slide records)
FIN, MAIN = tb.final_comparison(RES17, R2)
res = {'final_comparison_long.csv': compare_csv(FIN, FIN_DIR / 'final_comparison_long.csv') if (FIN_DIR / 'final_comparison_long.csv').exists() else (False, float('nan'), 'stored file not found'),
       'final_comparison_main_rand25.csv': compare_csv(MAIN, FIN_DIR / 'final_comparison_main_rand25.csv') if (FIN_DIR / 'final_comparison_main_rand25.csv').exists() else (False, float('nan'), 'stored file not found')}
report_compare(res)
print(MAIN.to_string(index=False))
FIN.to_csv(OUT_DIR / 'final_comparison_long.csv', index=False); MAIN.to_csv(OUT_DIR / 'final_comparison_main_rand25.csv', index=False)